# Nivel pandas 2

In [3]:
import pandas as pd
import random
import os

## Função que gera dados aleatórios

In [4]:
def gerar_dataframe_vendas_n2(n=120):

    clientes = ["Ana", "Bruno", "Carlos", "Daniela", "Eduardo", "Fernanda", "Gabriel", "Helena"]
    produtos = ["Mouse", "Teclado", "Monitor", "Headset", "Webcam", "Notebook"]
    categorias = {
        "Mouse": "Periféricos",
        "Teclado": "Periféricos",
        "Headset": "Periféricos",
        "Webcam": "Periféricos",
        "Monitor": "Hardware",
        "Notebook": "Hardware"
    }

    cidades = ["São Paulo", "Rio", "Curitiba", "Belo Horizonte", "Salvador"]

    dados = {
        "cliente": [random.choice(clientes) for _ in range(n)],
        "produto": [random.choice(produtos) for _ in range(n)],
        "cidade": [random.choice(cidades) for _ in range(n)],
        "quantidade": [random.randint(1, 5) for _ in range(n)],
        "preco_unitario": [random.randint(80, 3500) for _ in range(n)]
    }

    df = pd.DataFrame(dados)

    df["categoria"] = df["produto"].map(categorias)

    df["valor_total"] = df["quantidade"] * df["preco_unitario"]

    return df

In [5]:
df = None

if os.path.exists("dataframe_nivel_2.csv"):
    df = pd.read_csv("dataframe_nivel_2.csv", sep=",")
else:
    df = gerar_dataframe_vendas_n2()
    df.to_csv("dataframe_nivel_2.csv", sep=",", index=False)
df.head()

,cliente,produto,cidade,quantidade,preco_unitario,categoria,valor_total
0,Bruno,Webcam,Curitiba,3,932,Periféricos,2796
1,Bruno,Headset,Curitiba,5,553,Periféricos,2765
2,Ana,Teclado,Rio,4,674,Periféricos,2696
3,Daniela,Headset,Curitiba,2,2947,Periféricos,5894
4,Helena,Notebook,Salvador,1,3194,Hardware,3194


## Mostrando informações gerais do Df

In [6]:
df.describe()

,quantidade,preco_unitario,valor_total
count,120.000000,120.000000,120.000000
mean,2.825000,1855.091667,5221.650000
std,1.388201,979.169859,3977.816353
min,1.000000,80.000000,100.000000
25%,2.000000,989.000000,2036.500000
50%,3.000000,1996.500000,4112.000000
75%,4.000000,2728.000000,8185.000000
max,5.000000,3413.000000,17010.000000


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   cliente         120 non-null    str  
 1   produto         120 non-null    str  
 2   cidade          120 non-null    str  
 3   quantidade      120 non-null    int64
 4   preco_unitario  120 non-null    int64
 5   categoria       120 non-null    str  
 6   valor_total     120 non-null    int64
dtypes: int64(3), str(4)
memory usage: 6.7 KB


---

# Parte 1 - Análise por produto

## Quantidade total vendida por produto

In [8]:
df_quant_por_prod = df.groupby("produto")["quantidade"].sum()
df_quant_por_prod.head(10) # Como no jupyter não precisa usar o print eu vou optar por fazer dessa forma.

produto
Headset     62
Monitor     51
Mouse       66
Notebook    46
Teclado     68
Webcam      46
Name: quantidade, dtype: int64

## Faturamento total por produto

In [9]:
def real_format(data):
    return f"R${data:,.2f}".replace(",", "-").replace(".", ",").replace("-", ".")

df_fat_por_prod = df.groupby("produto")["valor_total"].sum()

# Fiz meio que na zueira isso, mas parece que ficou muito bom kk
# Obs: vi que la na frente eu vou usar mais então eu criei a função apartir do lambda.
df_fat_por_prod_format = df_fat_por_prod.map(
    # quis usar o lambda ao inves da função de propósito :D
    lambda x: f"R${x:,.2f}".replace(",", "-").replace(".", ",").replace("-", ".") 
    )

df_fat_por_prod_format.head(10)

produto
Headset     R$122.537,00
Monitor      R$81.608,00
Mouse       R$130.697,00
Notebook     R$84.604,00
Teclado     R$121.613,00
Webcam       R$85.539,00
Name: valor_total, dtype: str

## Top 3 produtos que mais geraram faturamento

In [10]:
df_top_3_prod_fat = df_fat_por_prod.sort_values(ascending=False)
df_top_3_prod_fat_format = df_top_3_prod_fat.map(real_format) # <- usei o map() me elogie kk
df_top_3_prod_fat_format.head(3)

produto
Mouse      R$130.697,00
Headset    R$122.537,00
Teclado    R$121.613,00
Name: valor_total, dtype: str

---

# Parte 2 - Análise por cidade

## Faturamento total por cidade

In [11]:
df_fat_por_cid = df.groupby("cidade")["valor_total"].sum()
df_fat_por_cid_format = df_fat_por_cid.map(real_format)
df_fat_por_cid_format.head(10)

cidade
Belo Horizonte    R$113.061,00
Curitiba           R$75.563,00
Rio               R$144.558,00
Salvador          R$153.754,00
São Paulo         R$139.662,00
Name: valor_total, dtype: str

## Cidade com maior faturamento

In [12]:
cid_maior_fat = df_fat_por_cid.idxmax() # Assim fica bem mais fácil usando o idxmax() e max()
fat_cid_maior_fat = df_fat_por_cid.max()
print(f"Cidade com maior faturamento: {cid_maior_fat}")
print(f"Faturamento da cidade: {real_format(fat_cid_maior_fat)}")

Cidade com maior faturamento: Salvador
Faturamento da cidade: R$153.754,00


## Média de valor total por venda em cada cidade

In [13]:
df_media_venda_por_cid = df.groupby("cidade")["valor_total"].mean()
df_media_venda_por_cid_format = df_media_venda_por_cid.map(real_format)
df_media_venda_por_cid_format.head()

cidade
Belo Horizonte    R$6.281,17
Curitiba          R$3.598,24
Rio               R$5.162,79
Salvador          R$5.913,62
São Paulo         R$5.172,67
Name: valor_total, dtype: str

---

# Parte 3 - Análise por categoria

## Quantidade total vendida por categoria

In [14]:
df_quant_por_cat = df.groupby("categoria")["quantidade"].sum()
df_quant_por_cat.head()

categoria
Hardware        97
Periféricos    242
Name: quantidade, dtype: int64

## Faturamento total por categoria

In [15]:
df_fat_por_cat = df.groupby("categoria")["valor_total"].sum()
df_fat_por_cat_format = df_fat_por_cat.map(real_format)
df_fat_por_cat_format.head()

categoria
Hardware       R$166.212,00
Periféricos    R$460.386,00
Name: valor_total, dtype: str

## Preço médio dos produtos por categoria

In [16]:
df_media_prod_por_cat = df.groupby("categoria")["preco_unitario"].mean()
df_media_prod_por_cat_format = df_media_prod_por_cat.map(real_format)
df_media_prod_por_cat_format.head()

categoria
Hardware       R$1.697,84
Periféricos    R$1.925,19
Name: preco_unitario, dtype: str

---

# Parte 4 - Múltiplas métrica (groupby avançado)

### Nesta eu travei por conta desse agg(), então gostaria de mais explicação sobre!

In [17]:
# Essa foi minha primeira tentativa
"""df_prod_geral = df.groupby("produto").agg({
    "quantidade":"sum",
    "preco_unitario":"mean",
    "valor_total":"sum",
    "produto":"count"
})"""

# Eu vi na documentação essa... Meio complicada né

df_prod_geral = df.groupby("produto").agg(
    Total_vendido=pd.NamedAgg(column="quantidade", aggfunc="sum"),
    Preco_medio=pd.NamedAgg(column="preco_unitario", aggfunc="mean"),
    Faturamento=pd.NamedAgg(column="valor_total",aggfunc="sum"),
    Vendas=pd.NamedAgg(column="produto", aggfunc="count")
)

df_prod_geral_format = df_prod_geral.copy() 
df_prod_geral_format["Preco_medio"] = df_prod_geral["Preco_medio"].map(real_format)
df_prod_geral_format["Faturamento"] = df_prod_geral["Faturamento"].map(real_format)

df_prod_geral_format.head(10)

,Total_vendido,Preco_medio,Faturamento,Vendas
produto,,,,
Headset,62,"R$1.945,58","R$122.537,00",24
Monitor,51,"R$1.657,11","R$81.608,00",19
Mouse,66,"R$1.969,18","R$130.697,00",22
Notebook,46,"R$1.740,83","R$84.604,00",18
Teclado,68,"R$1.800,30","R$121.613,00",20
Webcam,46,"R$1.986,41","R$85.539,00",17


---

# Parte 5 - Ranking

## Clientes que mais geram faturamento

In [18]:
df_top_5_clientes = df.groupby("cliente")["valor_total"].sum()
df_top_5_clientes_format = df_top_5_clientes.sort_values(ascending=False).map(real_format)
df_top_5_clientes_format.head(5)

cliente
Eduardo    R$112.653,00
Carlos      R$87.517,00
Ana         R$82.877,00
Daniela     R$75.689,00
Gabriel     R$73.793,00
Name: valor_total, dtype: str

---

# Desafio - Extra

In [19]:
df_desafio = df.groupby(["cidade", "produto"])["quantidade"].sum().sort_values(ascending=False)


df_desafio.groupby("cidade").head(1)

cidade          produto 
Rio             Teclado     26
Salvador        Notebook    23
Curitiba        Headset     22
São Paulo       Teclado     21
Belo Horizonte  Teclado     15
Name: quantidade, dtype: int64